<a href="https://colab.research.google.com/github/diptanu007/GenAI_Projects/blob/main/Langchain_agent_prompt_quality_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [2]:
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [13]:
!pip install langchain_google_genai

In [4]:
!pip install -U langchain langchain-community langchain-openai

In [1]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent

# 1. Setup Environment and LLM
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Define the Tool
@tool
def evaluate_prompt(user_prompt: str) -> str:
    """
    Evaluate a user's prompt based on clarity, specificity, context, persona, and output constraints.
    Assigns a score 1-10 and provides 3 suggestions for improvement.
    """
    eval_prompt_template = ChatPromptTemplate.from_template(
        """
        Evaluate the following user prompt.
        Return a JSON object with the following keys:
        - "Final_Score": integer between 1-10
        - "Justification": brief explanation
        - "Suggestions": list of 3 suggestions for improvement

        User_Prompt: {user_prompt}
        """
    )
    chain = eval_prompt_template | judge_llm
    result = chain.invoke({"user_prompt": user_prompt})
    return str(result.content)

# 3. Create the Agent
agent = create_agent(
    model=judge_llm,
    tools=[evaluate_prompt]
)

# 4. Helper and Test
def run_agent(input_str):
    response = agent.invoke({"messages": [("user", input_str)]})
    return response["messages"][-1].content

print(run_agent("Evaluate this prompt: I would like to improve my tennis serve.I am a beginner and playing tennis for 2 years.What drills are recommended."))

The evaluation of your prompt is as follows:

**Final Score:** 7

**Justification:** Your prompt is clear and specific about your goal to improve your tennis serve, and it provides relevant background information about your experience level. However, it could benefit from more detail regarding your current serve technique or specific areas where you struggle.

**Suggestions for Improvement:**
1. Include specific challenges you face with your serve to get more tailored advice.
2. Mention any equipment you have access to, such as a ball machine or a partner for practice.
3. Ask for recommendations on resources, such as videos or books, that can help you understand serve mechanics better.
